# COMPUTACIÓN CUÁNTICA - OPTATIVA - GRADOS ING. INFORMÁTICA Y SOFTWARE

# _Notebook_ sobre vectores y operadores (matrices) 


## Entorno de trabajo 

In [ ]:
# No olvidar que en "google colab" hay que instalar qiskit


# Instala la última versión disponible en collab (2.1)
try:
    import google.colab
    print("In colab, let's install things...")
    #
    !pip install qiskit==2.1.2 qiskit[visualization] qiskit-aer qiskit-ibm-runtime qiskit-optimization matplotlib numpy scipy sympy pandas
except ImportError:
    print("NOT in colab")

In [ ]:
!which conda && conda env list

In [ ]:
# Print package versions
import importlib
packages = [
    "qiskit",
    "qiskit_aer",
    "qiskit_ibm_runtime",
    "qiskit_optimization",
    "qiskit_machine_learning",
    "qiskit_nature",
]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:25s} : {mod.__version__}")
    except ImportError:
        print("%-25s: %s"%(pkg, "not installed"))


## Paquetería

In [ ]:
import numpy as np
import scipy.linalg as la
from IPython.display import display,Markdown,Latex
import matplotlib.pyplot as plt
import copy 

from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Statevector


In [ ]:
# Alguna configuración extra

# Esto es para que al imprimir se redondee y ocultar errores numéricos 
np.set_printoptions(suppress=True, precision=5)

## Utilería

In [ ]:
#
# RUTINAS para dibujar un número complejo como fasor
# Basadas en el material disponible en 
# https://talentq-es.github.io/TalentQ-CICC/docs/index.html
#
def plot_2D_plane(right=1,up=1,left=-1,down=-1,fsize=(8,8)):
    hpoints, vpoints = [],[]
    for i in range(left,right+1):
        if i!=0: hpoints.append(i)
    for i in range(down,up+1):
        if i!=0: vpoints.append(i)
    
    ax = plt.figure(figsize=fsize).gca()

    # Establece escalas iguales para ambos ejes
    ax.set(xlim=(left-0.5,right+0.5), ylim=(down-0.5, up+0.5), aspect='equal')
    # Elimina los bordes superior e inferior
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Establece los bordes inferior e izquierdo como ejes de coordenadas
    ax.spines['bottom'].set_position('zero')
    ax.spines['left'].set_position('zero')
    # Crea ticks en cada entero para habilitar dibujo de un retículo menor
    # Create minor ticks placed at each integer to enable drawing of minor grid
    ax.set_xticks(hpoints)
    ax.set_yticks(vpoints)
    # Dibuja retículo
    ax.grid(which='both', color='green', linewidth=1, linestyle='-', alpha=0.2)
    # Crea etiquetas 'x' e 'y' al final de cada eje
    ax.set_xlabel('x', size=14, labelpad=-24, x=1.03)
    ax.set_ylabel('y', size=14, labelpad=-21, y=1.02, rotation=0)
    # Dibuja flechas
    arrow_fmt = dict(markersize=4, color='black', clip_on=False)
    ax.plot((1), (0), marker='>', transform=ax.get_yaxis_transform(), **arrow_fmt)
    ax.plot((0), (0), marker='<', transform=ax.get_yaxis_transform(), **arrow_fmt)
    ax.plot((0), (1), marker='^', transform=ax.get_xaxis_transform(), **arrow_fmt)
    ax.plot((0), (0), marker='v', transform=ax.get_xaxis_transform(), **arrow_fmt)


def draw_sides(x=1,y=2,side_color="b",lwidth=1):
    plt.arrow(x,0,0,y,color=side_color,linestyle="dotted",width=0.001*lwidth)
    plt.arrow(0,y,x,0,color=side_color,linestyle="dotted",width=0.001*lwidth)
        
def draw_vector(x=1,y=0,vname="v",show_name=False,vcolor="b",sides=False,side_color="b",lwidth=1):
    plt.quiver(0,0,x,y,scale=1,scale_units='xy',angles = 'xy',color=vcolor,width=0.008*lwidth)
    dx = x
    if y<0: dy=y-0.3
    else: dy = y+0.3
        
    if show_name:
        vector_name="$"+vname+"=("+str(x)+","+str(y)+")$"
        plt.text(dx,dy,vector_name,color=vcolor)
    
    if sides:
        draw_sides(x,y,side_color)
        
def plot_complex_number(z,show_name=False,vcolor="b",sides=False,side_color="b",lwidth=1.4, size_cm=10, hold=False):
     x = z.real
     y = z.imag
     size_in = size_cm / 2.54
     if not hold:
         plot_2D_plane(left=-int(abs(x))-1,right=int(abs(x))+1,up=int(abs(y))+1,down=-int(abs(y))-1,fsize=(size_in,size_in))
     draw_vector(x=x,y=y,vname=False,show_name=False,vcolor=vcolor,sides=False,side_color=side_color,lwidth=lwidth)

def plot_cplx(z, color="b", size_cm=10, hold=False):
    """
        z = complex number
        color = a color string
        size_cm = figure size in cm
        hold = if hold=True the complex is drawn in the same axis as the previous one 
    """
    plot_complex_number(z, vcolor=color, size_cm=size_cm, hold=hold);

## Vectores

In [ ]:
# Generemos un objeto nparray de numpy representando un ket
# Recuerda: un ket es un vector columna
uket=np.array([[1 + 1.j],
               [2-3*1.j]])
 
# Varias formas de imprimir el vector
print("uket=\n",  uket)                                  # Formato de lista python
# Pretty print
display(array_to_latex(uket, prefix =  r'|u \rangle =')) # Latex (via método qiskit)
Statevector(uket).draw('latex', prefix=r'|u \rangle =')  # Qiskit usando base computacional

In [ ]:
# Para el ket anterior, representemos su bra asociado
display(array_to_latex(uket, prefix =  r'|u \rangle ='))

# Recuerda el bra es un vector fila formado por las componentes
# conjugadas complejas del ket (es decir, se obtiene conjugando
# y trasponiendo el vector columna)
ubra=uket.conj().T

display(array_to_latex(ubra, prefix =  r'\langle u| =') )

In [ ]:
# Observa la tupla de las dimensiones (shape) de los objetos
print("ket |u> shape=", uket.shape)
print("bra <u| shape=", ubra.shape)

# Podemos generar el bra a partir del ket cambiando su "shape" y
# conjugando
ubra2 = uket.reshape(1,2).conj()
display(array_to_latex(ubra2, prefix =  r'\langle u| =')) # ket->bra via reshape


In [ ]:
# Un ket con cuatro componentes 
uket=np.array([[ 1.+1.j],
               [ 2.-3.j],
               [ 2.+2.j],
               [-1.-1.j]])
display(array_to_latex(uket))
Statevector(uket).draw('latex')

<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> 
- En la celda anterior construye un ket con un número que no sea potencia de dos, por ejemplo 3 componentes, y observa su visualización como vector de estado con respecto a una dimensión potencia de dos
</div>

In [ ]:
######## SOLUCION
# Un ket con tres componentes 
uket=np.array([[ 1.+1.j],
               [ 2.-3.j],
               [-1.-1.j]])
display(array_to_latex(uket))
Statevector(uket).draw('latex')

<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> 
Define dos vectores de estado de dimensión 4, con parte imaginaria no nula, y determina:
 - su producto interno usando producto de matrices (bra x ket = vector fila x vector columna); no olvidar que el producto escalar es un escalar, esto es un número complejo, no una matriz 1x1
 - su producto interno usando ```np.dot()```
 - su producto interno usando ```np.vdot()```
</div>

In [ ]:
##### SOLUCIÓN

uket1=np.array([[ 1.+9.j], [ 2.-3.j],[ 2.+2.j],[-1.-17.j]])
uket2=np.array([[ 3     ], [ 5.-1.j],[ -2.   ],[   -1.j]])

# Dos formas de calcular el producto escalar <u2|u1>

# Producto de matrices
ubra2 = uket2.conj().T  # bra = adjunto = conjugado transpuesto del ket
s1 = ubra2 @ uket1 
s1 = s1[0][0]  # queremos un escalar no una matriz de 1x1
print("s1=",s1)

# Con el método np.dot(), hay que CONJUGAR, 
# pero no hace falta trasponer (si se hace no pasa nada) ¿por qué?
# y ¿porque se ha empleado el método np.flatten()?
s2 = np.dot(uket2.flatten().conj(), uket1.flatten())  # NO olvidar CONJUGAR el primer vector 
print("s2=",s2)

# Solución para vagos: np.vdot()
s3 = np.vdot(uket2,uket1)
print("s3=", s3)


In [ ]:
# Producto escalar y norma
uket=np.array([[ 1.+1.j], [ 2.-3.j],[ 2.+2.j],[-1.-1.j]])
ubra = uket.conj().T

m1 = abs(uket)
m2 = la.norm(uket)
m3 = np.sqrt(ubra @ uket)

<div style="border:1pt black solid; padding:3pt;">
    
**Ejercicio**

Imprime y analiza los resultados de la celda anterior y el tipo de dato obtenido. En caso de ser un array numpy imprime su shape. 
**Observa tanto el resultado como el tipo de datos del objeto obtenido y su dimensión**.
 - ¿Cuál se corresponde con la norma?
 - ¿Qué resultado se ha contenido en $m1$?
 - ¿Cómo obtener el escalar correspondiente a la norma a partir de $m3$?

</details>

In [ ]:
##### SOLUCIÓN

print(type(m1))
print(m1)
print("---")
print(type(m2))
print(m2)
print("---")
print(type(m3))
print(m3)

# Obtener el escalar asociado a m3
print("---")
m3ca = m3.flatten() # Una manera
m3cb = m3[0][0]     # Otra manera
print(m3ca, m3cb)

## Indexación de matrices

El siguiente ejercicio está orientado a familizarizarse con:
  - la base computacional de dimensión $n$
  - el significado de $A|{k}\rangle$
  - el significado de $\langle{i}|A|{j}\rangle$

In [ ]:
# Dimensión del espacio vectorial
n=3

# Generar una lista con los vectores (kets) de la base computacional
# Por ejemplo, los vectores de una base de dimensión 4: |0>, |1>, |2>, |3>

base = [np.zeros((n,1)) for i in range(0,n)] # inicializar una lista de vectores columna todos de cero
for k in range(0,n): # para generar la base, en el vector k-ésimo ponemos un 1 en la componente k-ésima
    base[k][k][0]=1
    display(array_to_latex(base[k], prefix =  r'|%i\rangle ='%k) )

# Generar matriz con los números enteros de 0 a n*n-1 permutados aleatoriamente 
# sumados con una componente imaginaria que aleatoriamente puede ser -1j, 0 o +1j
M = np.random.permutation(range(0,n*n)).reshape((n,n)) + np.random.randint(-1,2,(n,n))*1j
display(array_to_latex(M, prefix =  r'M=') )

# Calcular |v> = M |k> con k=0..n-1
for k in range(0,n):
    v = M @ base[k]
    display(array_to_latex(v, prefix =  r'|v_%i\rangle = M |%i\rangle='%(k,k)) )
    
# Probar calcular e = <f| M |c> con f,c=0..n-1, imprimir la secuencia
for f in range(0,n):      #row
    for c in range(0,n):  #column
        e = base[f].transpose().conj() @ M @ base[c]
        display(array_to_latex(e, prefix =  r'\langle %i| M |%i\rangle='%(f,c)) ) # y aquí que se muestra
        print('   --> ', e[0][0])  # Observa esto, qué estamos imprimiendo, ¿por qué el indexado [0][0]?
    


## Producto externo 

El siguiente ejercicio está orientado a familizarizarse con el significado de $|i\rangle\langle j|$:

In [ ]:
# Utilizaremos la dimensión y la lista de vectores de la base computacional del ejercicio anterior

# Calcular |i><j| para i,j=0,1...n-1 
    
# Probar calcular e = <f| M |c> con f,c=0..n-1, imprimir la secuencia
for i in range(0,n):      #row
    for j in range(0,n):  #column
        p = base[i] @ base[j].transpose().conj()  # Producto externo |i><j|
        display(array_to_latex(p, prefix =  r'|%i\rangle  \langle%i|='%(i,j)) ) # y aquí que se muestra
        #print('   --> ', e[0][0])  # Observa esto, qué estamos imprimiendo, ¿por qué el indexado [0][0]?
    


In [ ]:
# Veamos como |k><k| se comporta como proyector

# Generemos un vector aleatorio |v> con componentes enteras reales e imaginarias
v = np.random.randint(-9,9,(n,1)) + 1j*np.random.randint(-3,3,(n,1))
display(array_to_latex(v, prefix =  r'\text{(vector aleatorio) }|v\rangle=' )) 

# Crear una lista con los proyectores sobre cada elemento de la base computadional Pk=|k><k| para k=0,1...n-1 
P=[]
for k in range(0,n):      #row
   Pk = base[k] @ base[k].transpose().conj()  # Pk=|k><k|, proyector sobre el elemento |k> de la base computacional
   P.append(Pk)                               # añadir Pk a la lista
   display(array_to_latex(P[k], prefix =  r'\text{Proyector sobre }|%i\rangle: P_{%i}=|%i\rangle  \langle%i|='%(k,k,k,k)) )
    
# Proyectar el vector aleatorio sobre cada proyector ortogonal
for k in range(0,n):
    vp = P[k] @ v  # |vp> = Pk|v> es la proyección 
    display(array_to_latex(vp, prefix =  r'P_%i | v \rangle='%(k)))

# Todos los proyectores deben sumar la matriz unitaria
Is = np.zeros((n,n)) # Iniciar matriz todo ceros
for k in range(0,n): # Evaluar la suma de los proyectores
    Is = Is + P[k]
# Imprimir el resultado
display(array_to_latex(Is, prefix =  r'\text{(Esto debe ser la matriz identidad) } \sum_{k=0}^{%i}P_k = \sum_{k=0}^{%i} |k\rangle \langle k|  ='%(n,n)))
    


<div style="border:1pt black solid; padding:3pt;">

<b>Ejercicio</b>:

Repite los ejercicio anteriores usando una base **ortornormal** que **no** sea la computacional.

Para ello genera una matriz aleatoria, $B$ con coeficientes con parte real y compleja (poner coeficientes pequeños entre -2 y 2 por ejemplo). Comprueba que su determinate no es nulo (para que las columnas sean vectores linealmente independiente). Si da cero (o cercano) vuelve a generarla otra vez.

Vamos a aplicar el método de ortogonalización de **Gram-Schmidt** a los vectores columnas de esta matriz.

Esto es muy fácil de hacer usando la descomposición QR. La descomposición QR es un procedimiento de factorización de la matriz en otras dos matrices, $Q$ y $R$ ($B = Q R $) donde la matriz Q es una matriz ortonormal; por tanto, sus columnas son vertores ortonormales (tal como aplicando Gram-Schmidt). 

Puedes comprobar que $QQ^{\dagger}=I$.

Este es un ejemplo de como generar dicha base ortonormal aleatoria:

```python
    # Dimensión del espacio vectorial
    n=3
    
    # Generar una base ortogonal aleatoria:
    #   - generar una matriz cuadrada, B, con columnas linealmente independientes (det(B)!=0)
    #   - usar la descomposición QR 
    #   - las columnas de Q son una base ortonormal, es como aplicar Gram-Schmidt
    
    B = np.zeros((n,n))
    while la.det(B) == 0:
        B = np.random.randint(-1,3, (n,n)) + 1j * np.random.randint(-1,2,(n,n))
    display(B)
    
    # Gram-Schmidt via descomposición QR
    Q,R = la.qr(B)
    display(Q)

    # Esto es una comprobación de que Q*Qt debe ser identidad (matriz unitaria = sus columnas base ortonormal)
    display(Latex(r'$Q Q^t=$'), np.round(Q @ Q.conj().T, 3))
    
    c0 = Q[:,[0]] # Así se selecciona el primer vector columna
    display(array_to_latex(c0, prefix=r'|0_{\text{new base}}>='))

```
 
</div>

In [ ]:
##### SOLUCIÓN

# Dimensión del espacio vectorial
n=2

# Generar una base ortogonal aleatoria:
#   - generar una matriz cuadrada, B, con columnas linealmente independientes (det(B)!=0)
#   - usar la descomposición QR
#   - las columnas de Q son una base ortonormal, es como aplicar Gram-Schmidt

B = np.zeros((n,n))
while la.det(B) == 0:
    B = np.random.randint(1,8, (n,n)) + 1j * np.random.randint(-4,5,(n,n))
display("B=", B)

# Gram-Schmidt via descomposición QR
Q,R = la.qr(B)
display("Q=", np.round(Q,2))

# Esto es una comprobación de que Q*Qt debe ser identidad (matriz unitaria = sus columnas base ortonormal)
display(Latex(r'$Q Q^t=$'), np.round(Q @ Q.conj().T, 3))

c0 = Q[:,[0]] # Así se selecciona el primer vector columna
display(array_to_latex(c0, prefix=r'|0_{\text{new base}}>='))

#----

display(Markdown("---"), Markdown(r"**¡Atencíon!** A partir de aquí $|0\rangle, |1\rangle ...$ se refieren a los vectores de la nueva base"))


# Generar una lista con los vectores (kets) de la base computacional
# Por ejemplo, los vectores de una base de dimensión 4: |0>, |1>, |2>, |3>

base = [np.zeros((n,1)) for i in range(0,n)] # inicializar una lista de vectores columna todos de cero

# Tomar la base como las columnas de Q de la descompsición QR 
for k in range(0,n): # para generar la base, en el vector k-ésimo ponemos un 1 en la componente k-ésima
    base[k] = Q[:,[k]]
    display(array_to_latex(base[k], prefix =  r'|%i\rangle ='%k) )

# Generar matriz con los números enteros de 0 a n*n-1 permutados aleatoriamente 
# sumados con una componente imaginaria que aleatoriamente puede ser -1j, 0 o +1j
M = np.random.permutation(range(0,n*n)).reshape((n,n)) + np.random.randint(-1,2,(n,n))*1j
display(array_to_latex(M, prefix =  r'M=') )

# Calcular |v> = M |k> con k=0..n-1
for k in range(0,n):
    v = M @ base[k]
    display(array_to_latex(v, prefix =  r'|v_%i\rangle = M |%i\rangle='%(k,k)) )
    
# Probar calcular e = <f| M |c> con f,c=0..n-1, imprimir la secuencia
for f in range(0,n):      #row
    for c in range(0,n):  #column
        e = base[f].transpose().conj() @ M @ base[c]
        display(array_to_latex(e, prefix =  r'\langle %i| M |%i\rangle='%(f,c)) ) # y aquí que se muestra
        print('   --> ', e[0][0])  # Observa esto, qué estamos imprimiendo, ¿por qué el indexado [0][0]?
    
# --------------

# Veamos como |k><k| se comporta como proyector

# Generemos un vector aleatorio |v> con componentes enteras reales e imaginarias
v = np.random.randint(-9,9,(n,1)) + 1j*np.random.randint(-3,3,(n,1))
display(array_to_latex(v, prefix =  r'\text{(vector aleatorio) }|v\rangle=' )) 

# Crear una lista con los proyectores sobre cada elemento de la base computadional Pk=|k><k| para k=0,1...n-1 
P=[]
for k in range(0,n):      #row
   Pk = base[k] @ base[k].transpose().conj()  # Pk=|k><k|, proyector sobre el elemento |k> de la base computacional
   P.append(Pk)                               # añadir Pk a la lista
   display(array_to_latex(P[k], prefix =  r'\text{Proyector sobre }|%i\rangle: P_{%i}=|%i\rangle  \langle%i|='%(k,k,k,k)) )
    
# Proyectar el vector aleatorio sobre cada proyector ortogonal
for k in range(0,n):
    vp = P[k] @ v  # |vp> = Pk|v> es la proyección 
    display(array_to_latex(vp, prefix =  r'P_%i | v \rangle='%(k)))

# Todos los proyectores deben sumar la matriz unitaria
Is = np.zeros((n,n)) # Iniciar matriz todo ceros
for k in range(0,n): # Evaluar la suma de los proyectores
    Is = Is + P[k]
# Imprimir el resultado
display(array_to_latex(Is, prefix =  r'\text{(Esto debe ser la matriz identidad) } \sum_{k=0}^{%i}P_k = \sum_{k=0}^{%i} |k\rangle \langle k|  ='%(n,n)))
    


## Operadores lineales 

<b>Ejemplo:</b> 

Un <i>operador</i>  fácil de visualizar es el operador de <i>rotación en un plano</i>. Dado un ángulo $\theta \in (0,2\pi)$ el operador $A = R(\theta)$ gira cualquier vector un ángulo $\theta$ en el sentido antihorario

    
Un vector en el plano ${\bf u} =  (u_1,u_2)$  es equivalente al número complejo $u = u_1 + i u_2$ en el plano complejo $V = {\mathbb C}$.

    
Escrito en polares, $u=|u|e^{i\phi}$,  sabemos que una rotación de ángulo $\theta$ es equivalente a añadirle dicho  ángulo a la fase 
<br>
<br>
$$
 v = R(\theta) u = |u| e^{i(\phi + \theta)} =  |u| e^{i\phi } e^{i\theta} = u\cdot e^{i\theta} 
$$  
Por tanto, para rotar un número complejo un ángulo $\theta$ basta con multiplicarlo por la fase $e^{i\theta}$, que se corresponde con el operador $R(\theta)$ en el espacio vectorial $V = \mathbb{C}$.   La propiedad fundamental de una rotación es la de mantener invariante el módulo  $|v| = |u|$.    


In [ ]:
# Ejemplo de operador: rotación en el espacio euclídeo 2D 

# Ángulo que queremos rotar
# Probar diferentes ángulos
theta=np.pi/4

# vector a rotar (ket)
u=np.array([[3], [2]])  # vector columna
display(array_to_latex(u, prefix=r'|u\rangle='))

# Función que rota un vector tratándolo como un número complejo'
def R(u, theta):
        u0 = u[0][0]
        u1 = u[1][0]
        ucomp = u0 + 1j*u1 # Coger las componentes y tratarlo como un complejo
        vcomp = ucomp * np.exp(1j*theta) # u como complejo rotado un angulo theta
                                     # básicamente (multiplicar por e^(i theta))
        # Lo retornamos como vector ket
        vket = np.array( [[vcomp.real],
                          [vcomp.imag]])
        return vket 
#     
# Aplicar rotación 
v =  R(u, theta)
display(array_to_latex(v, prefix=r'|v\rangle='))
 
# Las normas de u y v deben ser iguales 
unorm = la.norm(u) 
vnorm = la.norm(v) 
display(Latex(rf'El módulo debe ser igual: $| |u\rangle |={unorm} ~~~ ||v\rangle|={vnorm}$'))
# plot_2D_plane(left=-int(abs(v1))-2,right=int(abs(v1))+2,up=int(abs(v2))+1,down=-int(abs(v2))-1)
# draw_vector(u1,u1,vcolor='b')
# draw_vector(v1,v2,vcolor='r')

# Representación de los vectores (comos si fueran complejos)
plot_cplx(unorm-1j*unorm, "none", size_cm=10) # Dirty trick: set axis as big as the biggest
plot_cplx(u[0][0] + 1j*u[1][0], "blue",   hold=True)
plot_cplx(v[0][0] + 1j*v[1][0], "orange",   hold=True)


**Ejemplo (continuación)**
    
Continuando con el ejemplo del operador de rotación en el espacio euclídeo 2D (el plano), hemos implementado la rotación convirtiendo los vectores en números complejos y básandonos en el hecho de que las componentes de esos complejos asociados a los vectores, $u = u_0 + i u_1$ y las de $R(\theta)\,u = v = v_0 + i v_0$ se obtienen mediante la multiplicación por una fase pura:

$$
\begin{align*}
v & = u e^{i\theta} \\
\end{align*}
$$
    
Vamos a desarrollar cada miembro en cartesianas, separando las partes real e imaginaria
<br>    

$$
\begin{align*}
v_0 + i v_1 &= (u_0 + iu_1) (\cos \theta + i \sin \theta)  \\
    &= (\cos\theta \, u_0 - \sin \theta\,  u_1) + i(\sin\theta\,  u_0 + \cos \theta\,  u_1)
\end{align*}
$$
<br>
    
es decir las coordenadas del vector origen y el vector rotado imagen se relacionan en la  forma 

<br>

$$
\begin{align*}
v_0 = \cos\theta \, u_0 - \sin \theta\,  u_1 ~~,~~
v_1 = \sin\theta \, u_0 + \cos \theta\,  u_1     
\end{align*}
$$

<br>

que podemos expresar en forma matricial
    
$$
\begin{pmatrix} v_0 \\ v_1 \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta &\cos\theta\end{pmatrix} \begin{pmatrix} u
_0 \\ u_1 \end{pmatrix}
$$    
    
Observa que cada columna de la matriz asociada al operador, es la imagen de los elementos de la base, esto es, la primera columna es la imagen (rotación) de $\begin{bmatrix} 1 \\ 0 \end{bmatrix}$ y la segunda columna  es la imagen (rotación) de $\begin{bmatrix} 0 \\ 1 \end{bmatrix}$.
    

<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> 
- Redefine la funcion anterior, de rotación en el plano, ```R(u,theta)```, pero aplicando la definción matricial de operador, es decir,
  $$ |v\rangle = A(\theta) \, |u\rangle $$
  donde A es la matriz anterior de rotación.
  $$
  A(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta &\cos\theta\end{bmatrix}
  $$    
Comprueba que los resultados son idénticos al ejemplo anterior que usaba números complejos equivalentes. 

- Calcula las imágenes de los vectores de la base computacional $|0\rangle$, $|1\rangle$. 
</div>

In [ ]:
##### SOLUCIÓN

# Función que devuelve la matriz asociada al operador de rotación 2D un ángulo theta 
def Rmat(theta):
        R = np.array([[np.cos(theta), -np.sin(theta)],
                      [np.sin(theta), np.cos(theta)]])
        return R 

# Función que rota un vector usando la definición 
def R(u, theta):
        R = Rmat(theta) 
        display(array_to_latex(R, prefix=rf'R(\theta)='))
        vket = R @ u
        return vket 
#     
# Aplicar rotación 
v =  R(u, theta)
display(array_to_latex(v, prefix=r'|v\rangle='))
 
# Las normas de u y v deben ser iguales 
unorm = la.norm(u) 
vnorm = la.norm(v) 
display(Latex(rf'El módulo debe ser igual: $| |u\rangle |={unorm} ~~~ ||v\rangle|={vnorm}$'))
# plot_2D_plane(left=-int(abs(v1))-2,right=int(abs(v1))+2,up=int(abs(v2))+1,down=-int(abs(v2))-1)
# draw_vector(u1,u1,vcolor='b')
# draw_vector(v1,v2,vcolor='r')

# Representación de los vectores (comos si fueran complejos)
plot_cplx(unorm-1j*unorm, "none", size_cm=10) # Dirty trick: set axis as big as the biggest
plot_cplx(u[0][0] + 1j*u[1][0], "blue",   hold=True)
plot_cplx(v[0][0] + 1j*v[1][0], "orange",   hold=True)

# Base computacional
I = np.eye(2)
b0 = I[:,[0]] # Primera columna de la identidad
b1 = I[:,[1]] # Segunda columna de la identidad

display(array_to_latex(b0, prefix=r'|b_0\rangle='))
display(array_to_latex(Rmat(theta) @ b0, prefix=r'R(\theta)\, |b_0\rangle='))
display(array_to_latex(Rmat(theta) @ b1, prefix=r'R(\theta)\, |b_1\rangle='))



## Proyección sobre un vector

In [ ]:
# Función auxiliar para realizar la operacion |u><v| siendo |u> y |v> vectores columna (shape (1,d))
    
def ket_bra(u,v):
    assert len(u)==len(v)
    return u @ v.conjugate().transpose()

def random_ket(d):
    return np.random.randint(-2,3,(d,1)) + 1j*np.random.randint(-1,2,(d,1))

def random_unit_ket(d):
    u = random_ket(d) # Vector aleatorio
    u = u / la.norm(u.flatten()) # Normalizar (dividir por el módulo)
                                 # Observa el uso del método flatten(), ¿por qué se usa aquí?
    return u


In [ ]:
# Dimensión para este ejemplo
d = 3

# generar un vector unitario aleatorio |u> 
u = random_unit_ket(d)
display(array_to_latex(u, prefix=r'|u\rangle='))
display(Latex(r'|u|=%f'%round(la.norm(u.flatten()),3)))

# construir proyector paralelo y perpendicular
P_par = ket_bra(u,u);
P_perp = np.identity(d) - P_par

display(array_to_latex(P_par, prefix=r'P_{par}='))
display(array_to_latex(P_perp, prefix=r'P_{perp}='))

In [ ]:
# Observa que estos tres resultados deben dar la matriz nula

# Por definición de proyector, P^2 = P, para el proyector paralelo
Z1 = P_par @ P_par - P_par
display(array_to_latex(Z1, prefix=r'Z1='))

# Por definición de proyector, P^2 = P para el proyector perpendicular 
Z2 = P_perp @ P_perp - P_perp
display(array_to_latex(Z2, prefix=r'Z2='))

# Los proyectores paralelo y perpendicular son ortogonales entre sí
Z3 = P_par @ P_perp
display(array_to_latex(Z3, prefix=r'Z3='))

In [ ]:
# Obtener las proyecciones paralela de otro vector aleatorio |v> (no tiene que ser unitario)
# sobre el vector unitario |u> anterior 
v = random_ket(d)

# Aquí se presentan tres formas de computar dichas proyecciones, observa el código
# y la notación de Dirac asociada
# Comprueba que son equivalentes

v_par = np.dot(P_par,v)
v_perp = np.dot(P_perp,v)

display(array_to_latex(v_par, prefix=r'P_{par}|v\rangle='))
display(array_to_latex(v_perp, prefix=r'P_{perp}|v\rangle='))
print("---")

v_par = P_par @ v
v_perp = P_perp @ v

display(array_to_latex(v_par, prefix=r'P_{par}|v\rangle='))
display(array_to_latex(v_perp, prefix=r'P_{perp}|v\rangle='))
print("---")

v_par = (u.conjugate().T @ v) * u   #(<u|v>)|u>
v_perp = v - v_par  #|v> - (<u|v>)|u>

display(array_to_latex(v_par, prefix=r'\langle u| v \rangle|v\rangle='))
display(array_to_latex(v_perp, prefix=r'|v\rangle - \langle u| v \rangle|v\rangle='))
print("---")


<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> <i>reflector</i>

Para el vector $|u\rangle$ del ejercicio anterior construye e imprime:
- el operador $R_u^{\perp}$ que <i>refleja</i> la componente perpendicular a $|{u}\rangle$ de cualquier  vector $|{\psi}\rangle$   
- el operador $R_{u}^{\|}$ que <i>refleja</i> la componente paralela a $|{u}\rangle$ de cualquier  vector $|{\psi}\rangle$  
</div>

In [ ]:
##### SOLUCIÓN

### Completar esta celda con la solución al ejercicio

## Autovalores y autovectores

In [ ]:
### Dimensión del espacio de trabajo
n = 2

# Generemos una matriz aleatoria NO normal A At != At A
# Para ello iteramos hasta que demos con una matriz que no cumple 
# la condicion de normal
while True:
    A = np.random.rand(n,n) + 1j*np.random.rand(n,n)  # Coeficientes aleatorios en [0,1]
    A = np.round(3*A, 0) # Escalar a 3, y redondeamos
    # Chequear que no es normal: A At != At A
    At= A.conj().T  # Adjunta 
    AAt = A @ At
    AtA = At @ A
    # Comprobar si dos matrices son iguales con una tolerancia
    if not np.allclose(AAt, AtA, atol=1e-3, rtol=1e-3): # buscamos una NO normal
        break

display(array_to_latex(A, prefix=r'A='))
display(array_to_latex(AAt-AtA, prefix=r'\text{(esto NO debe ser cero si A NO normal) } A A^{\dagger} - A^{\dagger} A ='))

# En lugar de una matriz aleatoria prueba otras matrices sencillas
# A = np.array([[3,2],[1,2]]) # Esta es diagonalizable
# A = np.array([[1,1],[0,1]]) # Esta no es diagonalizable
 
# Cálculo de autovalores y autovectores
eigvals, eigvecs = np.linalg.eig(A)
print('lista de autovalores =',eigvals)
print('matriz de autovectores =', "\n", eigvecs)

# OBSERVA que la matriz de autovalores es una doble lista python
# donde eigvecs[:,k] (columna k-ésima) representa el k-ésimo autovector
# el que corresponde con el autovalor eigvals[k]
# Pero, ATENCIÓN: eigvecs[:,k] es una lista!
# Es decir, un objecto con 'shape' (n)
# No son vectores columna (tendrían shape=(n,1))

# Recorrer la lista de autovalores y autovectores e imprimirlos
for k in range(len(eigvals)):
    lambdak = eigvals[k]                             # Autovalor k-ésimo
    ketlambdak = np.array(eigvecs[:,k]).reshape(n,1) # Autovector k-ésimo como vector columna 
    display(Latex(rf'$\lambda_{k}={lambdak}$')) # Observa el uso de raw f-strings r'$...$'
    display(array_to_latex(ketlambdak, prefix=rf'|\lambda_{k}\rangle='))



<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> 
- Construye la matriz de paso $P$ cuyas columnas son los autovectores
- Construye la matriz diagonal, $D$, cuyos elementos son los autovalores
- Comprueba si es diagonalizable, es decir: $A = P D P^{-1}$
</div>

In [ ]:
###########  SOLUCIÓN
D = np.diag(eigvals)
P = np.array(eigvecs)

M = P @ D @ la.inv(P) # Construimos M = P D P^-1
display(array_to_latex(P, prefix=r'P='))
display(array_to_latex(M, prefix=r'M='))

# Comprobar que A==M
display(array_to_latex(M-A, prefix=r'M-A='))


In [ ]:
# En este ejemplo:
#    - extraeremos la parte hermítica de la matriz anterior
#    - calcularemos su espectro de autovalores y autovectores
#    - comprobaremos algunas propiedades

# Crear una matriz hermítica (que será normal) a partir de la matriz A
H = 1/2 * (A+A.conj().T)
display(array_to_latex(H, prefix=r'H='))

# Chequear que realmente es normal  H Ht = Ht H
display(array_to_latex(H @H.conj().T - H.conj().T @ H, prefix=r'H H^{\dagger} - H^{\dagger} H=')) # Esto debe ser la matriz nula

# Calculemos sus autovalores y autovectores
eigvals, eigvecs = np.linalg.eig(H)
print('lista de autovalores =', eigvals, "<-- comprobar que son reales") 
print('matriz de autovectores =',"\n", eigvecs)

# Veamos que en este caso la matriz de paso es una matriz unitaria
U = np.matrix(eigvecs)  # convertir lista de lista python a nparray de numpy
display(array_to_latex(U, prefix=r'\text{ matriz de paso, debe ser unitaria } U='))

# Visualicemos que es unitaria
display(array_to_latex(U*U.conj().T, prefix=r'U U^{\dagger}='))

# Veamos su diagonalización
D = np.round(la.inv(U) @ H @ U, 8)   # D = U^-1 H U = Ut H U
display(array_to_latex(D, prefix=r'\text{ diagonalización de H: } D=U^{-1} H U= U^{\dagger} H U ='))


<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: Descomposición espectral</b>

La matriz anterior $H$ es hermítica, y por tanto normal. Así que admite descomposición espectral.

Escriba un bucle que recorra los diferentes autovalores, $|\lambda_k\rangle$ y calcule su proyector (imprimiendolo también).

Comprobar el teorema espectral en otro bucle, es decir que la suma de los proyectores multiplicados por su autovalor da como resultado la matriz $H$ original: $H = \sum_k \lambda_k  |{\lambda_k}\rangle \langle{\lambda_k}|$.

</div>

In [ ]:
#### SOLUCIÓN

P=[]
for k in range(len(eigvecs)):
    ketlambdak = np.array(eigvecs[:,k]).reshape(n,1)
    Pk = ketlambdak @ ketlambdak.conj().T
    P.append(Pk)
    display(array_to_latex(Pk, prefix=rf'P_{k}=|\lambda_{k}\rangle \langle \lambda_{k}|='))

# Descomposición espectral
S = np.zeros((n,n))
for k in range(n):
    S = S + P[k] * eigvals[k]
    
display(array_to_latex(S, prefix=r'S=\sum_k \lambda_k  |{\lambda_k}\rangle \langle{\lambda_k}|='))
display(array_to_latex(S-H, prefix=r'S-H='))



## Potencia y exponencial de una matriz (operador) 

**Potencia de una matriz (operador)**

Dada una matriz diagonal, una potencia entera de dicha matriz es otra matriz diagonal con las potencias de los elementos de la diagonal, por ejemplo,

$$
\begin{bmatrix}
a & 0 \\
0 & b 
\end{bmatrix}^n = 
\begin{bmatrix}
a^n & 0 \\
0 & b^n
\end{bmatrix} 
$$

Ello nos permite realizar fácilmente la potencia entera de una matriz diagonalizable, ya que sus autovalores serán las potencias de los autovalores de la matriz y los autovectores no cambian:
$$
A^n = (P D P^{-1})^n = P D P^{-1}\, P D P^{-1} ...^{n \text{ veces})} P D P ^{-1} = P D^n P^{-1}
$$ 

**Exponencial de un operador**

Cualquier función que podamos expandir en polinomio de Taylor podremos aplicar el resultado anterior y por tanto extender su aplicación a una matriz, **siempre que sea diagonalizable**:

$$
\exp(A) = e^A = I + A + \frac{1}{2} A^2 + \frac{1}{3!} A^3 + ... 
$$

Es decir la exponencial de una matriz diagonalizable, es otra matriz con los mismos autovectores pero cuyos autovalores son la exponecial de los autovalores de la matriz original:

$$ A = P D P^{-1} \Rightarrow e^A = P e^D P^{-1} \text{ con } 
e^D=\begin{bmatrix}
    e^{\lambda_0} & 0 & \cdots \\
    0 & e^{\lambda_1} & 0 &\cdots \\
    \vdots & \vdots & & \\
    0 & 0             & 0 & e^{\lambda_{k-1}}
    \end{bmatrix}
$$


In [ ]:
# Ilustremos la exponencial de una matriz continuando con la matriz A aleatoria generada anteriormente

# Usemos el método expm() de algebra lineal
eA = la.expm(A) # computa e^A

display(array_to_latex(A,prefix='A = '))
display(array_to_latex(np.round(eA,3),prefix='e^A = ')) # Redondear para mejor visualización 


<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> 
- Comprueba que $A$ y $e^A$ tienen los mismos autovectores
- Comprueba que los autovalores de $e^A$ son las exponenciales de los autovalores de $A$
</div>

In [ ]:
###### SOLUCIÓN

eigvalA, eigvecA = np.linalg.eig(A)
eigvalAe, eigvecAe = np.linalg.eig(eA)

print('lista de autovalores de A =',eigvalA)
print('matriz de autovectores de A =', "\n", eigvecA)

print('lista de autovalores de e^A =',eigvalAe)
print('matriz de autovectores de e^A =', "\n", eigvecAe)

diff= eigvalAe - np.exp(eigvalA) # Esto debe ser cero 
print('autovalores de e^A - exponecial autovalores de A (deb ser 0) = ', diff)


## Matrices de Pauli   

Se definen las matrices de Pauli 
<br>
$$
\sigma_x = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}~~~~~,~~~~~~~~~
\sigma_y = \begin{bmatrix} 0 & -i \\ i & 0 \end{bmatrix}~~~~~,~~~~~~~~~
\sigma_z = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix} \, .
$$


<b>Nota:</b> 
Se suele denotar también $\sigma_i$ donde $\sigma_1=\sigma_x, ~
\sigma_2=\sigma_y$  y  $\sigma_3=\sigma_z$. También $\boldsymbol{\sigma}$ denota el <i>vector</i> de matrices de Pauli $\boldsymbol{\sigma} = (\sigma_1,\sigma_2,\sigma_3)$ (esto es un vector de matrices). 

Si añadimos la matriz identidad $I = \sigma_0 = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}~
$  el conjunto $\{I,\sigma_x,\sigma_y,\sigma_z\}$ forma una *base* para el espacio de *matrices hermíticas* $2\times 2$ con coeficientes reales.  

$$
\begin{align*}
A = a_0 I + {\bf a} \cdot \boldsymbol{\sigma}  &=   a_0 I + a_1 \sigma_1 + a_2 \sigma_2 + a_3 \sigma_3  = 
\begin{bmatrix}
a_0 + a_3 & a_1 - i a_2 \\ a_1 + i a_2 & a_0 - a_3
\end{bmatrix} = A^\dagger
\end{align*}
$$

con $a_i\in {\mathbb R}$ cuatro números reales.

Propiedades destacables de las matrices de Pauli:

- son hermíticas $\sigma_i^\dagger = \sigma_i$

- son unitarias $\sigma_i^\dagger = \sigma_i^{-1} = \sigma_i$

- tienen traza nula $\text{tr}(\sigma_i) = 0$


<div style="border:1pt black solid; padding:3pt;">

<b> Ejercicio: </b> 
- Calcula los autovectores y autovalores de cada una de las matrices de Pauli, y también de la identidad 
- Chequea que los autovectores obtenidos están normalizados, y si no normalízalos
</div>

In [ ]:
##### SOLUCIÓN

# Esta función define las matrices de Pauli en un diccionario 
# direccionable por número o nombre
def get_pauli_matrices():
    """Genera las 4 matrices de Pauli (I, X, Y, Z)"""
    I = np.array([[1, 0], 
                  [0, 1]])
    X = np.array([[0, 1], 
                  [1, 0]])
    Y = np.array([[0, -1j], 
                  [1j, 0]])
    Z = np.array([[1, 0], 
                  [0, -1]])
    
    return I, X, Y, Z 

I, X, Y, Z = get_pauli_matrices()

eigval, eigvec = la.eig(I)
display(array_to_latex(I,prefix='I = '))
display(array_to_latex(eigval,prefix=r'\text{Autovalores de I}='))
display(array_to_latex(eigvec,prefix=r'\text{Autovectores de I}='))


eigval, eigvec = la.eig(X)
display(array_to_latex(X,prefix='X = '))
display(array_to_latex(eigval,prefix=r'\text{Autovalores de X}='))
display(array_to_latex(eigvec,prefix=r'\text{Autovectores de X}='))


eigval, eigvec = la.eig(Y)
display(array_to_latex(Y,prefix='Y = '))
display(array_to_latex(eigval,prefix=r'\text{Autovalores de Y}='))
display(array_to_latex(eigvec,prefix=r'\text{Autovectores de Y}='))


eigval, eigvec = la.eig(Z)
display(array_to_latex(Z,prefix='Z = '))
display(array_to_latex(eigval,prefix=r'\text{Autovalores de Z}='))
display(array_to_latex(eigvec,prefix=r'\text{Autovectores de Z}='))
